In [5]:
# hugging face rotten tomatoes dataset

from datasets import load_dataset

# The current HfUriError indicates a persistent issue with the installed versions
# of 'datasets' and 'huggingface_hub' libraries, as 'rotten_tomatoes' is a top-level dataset.
# A code-only fix is not directly apparent given the conflicting errors encountered.
# Reverting to the canonical way to load this dataset.
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
data

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [6]:
from transformers import pipeline

model_path = "cardiffnlp/twitter-roberta-base-sentiment"

pipe = pipeline(model = model_path,
                tokenizer=model_path,
                return_all_scores=True,
                device="cuda:0")

config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [7]:
data["train"][0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

In [17]:
#test model on test dataset

import numpy as np
from tqdm import tqdm

y_pred=[]
texts_to_process = list(data["test"]["text"])

# The pipeline, despite `return_all_scores=True`, appears to return a list of single top predictions (dictionaries)
# rather than a list of lists of all scores. We will process it as such.
pipeline_results = pipe(texts_to_process)

for op_item in tqdm(pipeline_results, total=len(texts_to_process)):
    # op_item is a dictionary like {'label': 'LABEL_X', 'score': Y}
    predicted_label = op_item['label']

    assingment = None
    if predicted_label == 'LABEL_0': # Negative
        assingment = 0
    elif predicted_label == 'LABEL_2': # Positive
        assingment = 1
    else: # predicted_label == 'LABEL_1' (Neutral)
        # Rotten Tomatoes dataset has binary labels (0: negative, 1: positive).
        # We need to map the model's neutral prediction to one of these binary categories.
        # Arbitrarily mapping neutral ('LABEL_1') to 0 (negative) for a binary classification context.
        assingment = 0 # Consider neutral as non-positive, thus mapping to 0 for binary comparison
    y_pred.append(assingment)

100%|██████████| 1066/1066 [00:00<00:00, 853431.58it/s]


In [19]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
  performance = classification_report(
      y_true, y_pred,
      target_names=["negative", "positive"]
  )
  print(performance)


In [20]:
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    negative       0.68      0.92      0.78       533
    positive       0.88      0.58      0.70       533

    accuracy                           0.75      1066
   macro avg       0.78      0.75      0.74      1066
weighted avg       0.78      0.75      0.74      1066



In [22]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

In [23]:
train_embeddings.shape


(8530, 768)

In [26]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=20)
clf.fit(train_embeddings, data["train"]["label"])

LogisticRegression(random_state=20)

In [27]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    negative       0.85      0.86      0.85       533
    positive       0.86      0.85      0.85       533

    accuracy                           0.85      1066
   macro avg       0.85      0.85      0.85      1066
weighted avg       0.85      0.85      0.85      1066

